# 06 — Multi-Subject IMU Traditional Machine Learning

In Notebook 05, we built hand-crafted IMU features and trained classical ML models on **one person's data**.

Now we have a much more interesting question:

> **Can a model trained on several people recognize activity in a person it has never seen?**

The blinded class dataset contains multiple anonymous subjects:

```text
subject01
subject05
subject06
...
```

Each row/window still represents one 2-second IMU sample, but now every window also carries a **subject ID**.

Today we will:

1. load the blinded multi-subject dataset,
2. reuse the hand-crafted features from Notebook 05,
3. inspect activity and subject structure,
4. deliberately do a **bad random-window split**,
5. then do the correct **leave-one-subject-out** validation,
6. evaluate accuracy, macro F1, confusion matrix, and per-subject performance,
7. inspect Logistic Regression coefficients,
8. compare with a Random Forest.

The hidden final test subjects remain untouched.

> **Important:** `subject_id` is metadata for splitting. It is **not a predictor feature**. Otherwise the model can start learning *who* produced the data instead of *what activity* they were doing.


## 1. Load the blinded class dataset

The harmonized file should contain:

- `X_windows`: raw IMU windows, shape `(n_windows, 100, 6)`
- `y_windows`: activity labels
- `groups` / `subject_id`: anonymous subject ID for each window
- `window_id`: anonymous unique window identifier
- `channel_names`
- `sample_rate_hz`

The notebook searches a few likely locations so it works whether the file sits beside this notebook or inside the harmonization folder.


In [2]:
%matplotlib inline

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import (
    LeaveOneGroupOut,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_CANDIDATES = [
    Path("train_blinded_20261015.npz"),
    Path("train_blinded.npz"),
    Path("imu_harmonized_20261015/train_blinded_20261015.npz"),
    Path("STUDENT_SHARE_BUNDLE_20261015/train_blinded_20261015.npz")
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)

assert DATA_PATH is not None, (
    "Could not find the blinded training NPZ. "
    "Put train_blinded_20261015.npz beside this notebook."
)

data = np.load(DATA_PATH, allow_pickle=False)

# COMPLETE:
# Pull out X_windows, y_windows, subject_ids, groups, window_ids,
# CHANNELS, and FS from the NPZ.
#
# Remember:
#   .astype(str) is useful for the label/string arrays.
#   float(...) converts the 0-D sample_rate_hz array to a Python number.

# X_windows = ...
# y_windows = ...
# subject_ids = ...
# groups = ...
# window_ids = ...
# CHANNELS = ...
# FS = ...

print("Loaded:", DATA_PATH)
print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("Subjects:", np.unique(subject_ids))
print("Channels:", CHANNELS)
print("Sampling rate:", FS, "Hz")


Loaded: STUDENT_SHARE_BUNDLE_20261015/train_blinded_20261015.npz


NameError: name 'X_windows' is not defined

### Your turn

Before modeling, answer these from the printed output:

1. What does each dimension of `X_windows.shape` represent?
2. How many anonymous subjects are in the development dataset?
3. Why do we need `groups` if we already have `y_windows`?
4. Why would giving `subject_id` to the classifier as a predictor be a bad idea?


## 2. Inspect class balance **within each subject**

Overall class counts can hide a problem. A subject might have lots of Walking but very little StairsDown.

A cross-tabulation gives us:

```text
subject × activity → number of windows
```

This is also a nice first reminder that our observations are hierarchical:

```text
subject
   ↓
activity bout
   ↓
overlapping windows
```


In [ ]:
# COMPLETE:
# Make a pd.crosstab with subject_ids as rows and y_windows as columns.
# Add margins=True so you also get totals.
#
# Save it as subject_activity and display it.

# subject_activity = pd.crosstab(...)
# display(subject_activity)

# Then make a Series showing the number of windows per subject.
# Hint: pd.Series(subject_ids).value_counts().sort_index()


## 3. Eyeball raw windows across people

Before feature engineering, look at a few examples from **different subjects**.

The function below plots acceleration and gyroscope magnitude for the first matching window from selected subjects. We are not expecting identical waveforms. We are hoping the broad activity signal survives person-to-person variation.


In [ ]:
acc_indices = [CHANNELS.index(c) for c in ["ax", "ay", "az"]]
gyro_indices = [CHANNELS.index(c) for c in ["gx", "gy", "gz"]]

def vector_magnitude(window, indices):
    return np.sqrt(np.sum(window[:, indices] ** 2, axis=1))


def plot_activity_across_subjects(activity, max_subjects=3):
    t = np.arange(X_windows.shape[1]) / FS

    available_subjects = np.unique(
        subject_ids[y_windows == activity]
    )[:max_subjects]

    fig, ax = plt.subplots(figsize=(10, 4))

    for subject in available_subjects:
        candidates = np.where(
            (subject_ids == subject)
            & (y_windows == activity)
        )[0]

        idx = candidates[len(candidates) // 2]
        amag = vector_magnitude(
            X_windows[idx],
            acc_indices,
        )

        ax.plot(
            t,
            amag,
            label=subject,
            alpha=0.8,
        )

    ax.set_xlabel("Time within window (s)")
    ax.set_ylabel("Acceleration magnitude")
    ax.set_title(
        f"{activity}: examples from different subjects"
    )
    ax.legend()
    plt.show()


for activity in np.unique(y_windows):
    plot_activity_across_subjects(activity)


### Your turn

Do examples from different people look identical? Good.

Which activity appears most consistent across subjects? Which seems most variable?

Think ahead: **person-to-person variability is exactly what the final held-out test set is testing.**


## 4. Reuse your hand-crafted features from Notebook 05

Software reuse is a feature, not cheating.

Bring forward the functions you already built:

- `basic_stats`
- `vector_magnitude`
- `spectral_features`
- `extract_hand_features`

We will calculate the deterministic hand features **once** for all windows.

### Leakage checkpoint

Is it okay to calculate these features before cross-validation?

**Yes**, because each feature here is calculated independently from one window and does not learn population-wide parameters.

But `StandardScaler`, PCA used for modeling, feature selection based on all labels, etc. **must be fitted inside each training fold**.

That distinction is important.


In [ ]:
# COMPLETE:
# Paste/adapt your four working functions from Notebook 05 here:
#
# basic_stats(...)
# vector_magnitude(...)
# spectral_features(...)
# extract_hand_features(...)
#
# Keep the same feature definitions you used in 05 so we are comparing
# the same model idea on a harder multi-subject problem.


# def basic_stats(...):
#     ...


# def vector_magnitude(...):
#     ...


# def spectral_features(...):
#     ...


# def extract_hand_features(...):
#     ...


## 5. Build one feature row per window

Now convert the raw `(100 × 6)` waveform window into an interpretable feature vector.

We keep `activity`, `subject_id`, and `window_id` beside the features as **metadata**, but they will not enter `X_features`.


In [ ]:
# COMPLETE:
# 1. Run extract_hand_features(...) on all X_windows.
# 2. Add activity, subject_id, and window_id columns.
# 3. Create metadata_columns = ["activity", "subject_id", "window_id"].
# 4. Build feature_columns automatically so metadata cannot accidentally
#    enter the model.
# 5. Create X_features with .to_numpy().

# feature_df = ...
# feature_df["activity"] = ...
# feature_df["subject_id"] = ...
# feature_df["window_id"] = ...

# metadata_columns = [...]

# feature_columns = [
#     ...
# ]

# X_features = ...

print("Raw windows:", X_windows.shape)
print("Feature matrix:", X_features.shape)
print("Number of features:", len(feature_columns))
display(feature_df.head())


### Your turn

Why would `subject_id` be dangerous as a predictor?

Suppose each person's phone placement or gait is slightly distinctive. A model given subject identity could exploit that shortcut instead of learning general activity structure.


## 6. Explore features across activities

The same features that looked wonderful within one person may overlap more once seven people are pooled.

That is not a bug. It is the real generalization problem arriving at the party.


In [ ]:
candidate_features = [
    "amag_std",
    "amag_range",
    "amag_dominant_frequency",
    "gmag_std",
    "gmag_dominant_frequency",
]

for feature in candidate_features:
    if feature not in feature_df.columns:
        continue

    activities = np.unique(y_windows)

    values_by_activity = [
        feature_df.loc[
            feature_df["activity"] == activity,
            feature,
        ]
        for activity in activities
    ]

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.boxplot(
        values_by_activity,
        tick_labels=activities,
    )

    ax.set_ylabel(feature)
    ax.set_title(
        f"{feature}: pooled across subjects"
    )

    plt.xticks(rotation=20)
    plt.show()


## 7. PCA sanity check: activity structure **and subject structure**

We will make two PCA views of the **same feature matrix**:

1. color by activity,
2. color by anonymous subject.

This is exploratory visualization only, not model validation.

Because PCA depends on population-wide variance, we do **not** use this PCA representation as input to our cross-validation model below. If we wanted PCA inside the model, it would belong inside the sklearn Pipeline so it is fitted separately in every training fold.


In [ ]:
# COMPLETE:
# Standardize X_features for visualization, then fit PCA(n_components=2).
#
# IMPORTANT: this PCA is ONLY for visualization.
# We are NOT going to feed these all-data PCA scores into cross-validation.

# pca_scaler = StandardScaler()
# X_scaled_for_plot = ...
# pca = PCA(...)
# X_pca = ...

print(
    "Variance explained by PC1 + PC2:",
    pca.explained_variance_ratio_.sum(),
)


# COMPLETE: make one scatter plot colored by activity.
# Loop over np.unique(y_windows), make a Boolean mask, and scatter PC1 vs PC2.


# COMPLETE: make a second scatter plot of THE SAME X_pca,
# but this time color/group points by subject_ids.


### Your turn

Does the feature space appear to contain subject-to-subject structure?

If yes, why does that make a random window split particularly dangerous?


## 8. First, deliberately do the **wrong** validation

This is the tempting approach:

```python
train_test_split(all_windows)
```

The split knows nothing about subjects. Windows from the same person, and possibly neighboring overlapping windows from the same activity bout, can land on both sides.

Let's calculate that score anyway so we can compare it with the honest one.


In [ ]:
# COMPLETE:
# Make a Pipeline called window_model:
#   StandardScaler()
#   LogisticRegression(max_iter=2000, class_weight="balanced")
#
# Then do a stratified 75/25 train_test_split on X_features and y_windows.
# Fit, predict, and calculate:
#   bad_accuracy
#   bad_macro_f1
#
# Give random_state=42 so everyone gets the same split.

# window_model = Pipeline([...])

# X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(...)

# window_model.fit(...)
# y_pred_bad = ...

# bad_accuracy = ...
# bad_macro_f1 = ...

print("BAD random-window validation")
print("Accuracy:", bad_accuracy)
print("Macro F1:", bad_macro_f1)


## 9. Correct validation: leave one **whole subject** out

Now we ask the real question.

`LeaveOneGroupOut()` creates folds like:

```text
Fold 1: train on subjects 05,06,07,...   test on subject01
Fold 2: train on subjects 01,06,07,...   test on subject05
...
```

Every prediction is therefore made for a subject the model did not see during fitting.

`cross_val_predict()` is especially useful here because it returns one honest held-out prediction for **every window**.


In [ ]:
# COMPLETE:
# Build subject_model with the SAME model recipe as above.
#
# Make:
#   logo = LeaveOneGroupOut()
#
# Use cross_val_predict with:
#   subject_model
#   X_features
#   y_windows
#   groups=subject_ids
#   cv=logo
#
# Save predictions as y_pred_grouped.
#
# Then calculate grouped_accuracy and grouped_macro_f1.

# subject_model = Pipeline([...])
# logo = LeaveOneGroupOut()

# y_pred_grouped = cross_val_predict(...)

# grouped_accuracy = ...
# grouped_macro_f1 = ...

print("CORRECT leave-one-subject-out validation")
print("Accuracy:", grouped_accuracy)
print("Macro F1:", grouped_macro_f1)

# COMPLETE:
# print a classification_report(...)


### Stop and compare

The random-window score answers:

> Can I recognize another window drawn from this mixed pool of already-seen people?

The grouped score answers:

> Can I recognize activity in a **new person**?

For our class project, the second question is the one that matters.


In [ ]:
# COMPLETE:
# Make a two-row DataFrame comparing:
#   Random windows (BAD)
#   Leave one subject out
#
# Include accuracy and macro_f1 columns.
# Then display it.

# comparison = pd.DataFrame({...})
# display(comparison)


## 10. Confusion matrix: **what** is the model confusing?

A single score can hide the biology.

Look for patterns such as:

- Walking → StairsDown
- StairsDown → Walking
- Standing → locomotion

Those are much more informative than just saying "63%".


In [ ]:
# COMPLETE:
# Use ConfusionMatrixDisplay.from_predictions(...)
# with the TRUE y_windows and the honest grouped predictions.
#
# Add a useful title and plt.show().


## 11. How different is performance across held-out subjects?

Because every `y_pred_grouped` prediction was generated while that subject was absent from training, we can safely calculate a score for each anonymous person.

This is often more revealing than one pooled number.


In [ ]:
# COMPLETE:
# Loop over np.unique(subject_ids).
#
# For each subject:
#   1. make a Boolean mask,
#   2. calculate accuracy on that subject,
#   3. calculate macro F1 on that subject,
#   4. append a dictionary to subject_scores.
#
# Convert to a DataFrame and sort by macro_f1.
#
# Then make a bar plot of subject_id vs macro_f1.

# subject_scores = []
# for subject in ...:
#     ...

# subject_scores = pd.DataFrame(...)

# display(subject_scores)

# fig, ax = plt.subplots(...)
# ax.bar(...)
# ...


### Your turn

The model can be quite good on one unseen person and poor on another.

What real-world differences could produce that?

Think about gait, body size, phone placement, clothing, annotation quality, pace, stairs technique, and sensor hardware.


## 12. Fit on all development subjects and inspect Logistic Regression coefficients

Cross-validation estimates generalization.

After evaluating the model recipe, we can fit that recipe once on **all available development subjects**. This is the model we would eventually use to predict the untouched final test set.

Because scaling is inside the Pipeline, the Logistic Regression coefficients correspond to standardized features.


In [ ]:
# COMPLETE:
# Build the final Logistic Regression pipeline with StandardScaler
# and LogisticRegression.
#
# Fit it on ALL X_features and y_windows.
#
# Pull the classifier out using .named_steps["classifier"].
#
# For each class, make a DataFrame containing:
#   feature
#   coefficient
#   abs_coefficient
#
# Sort by abs_coefficient descending and display the top 10.

# final_logit_model = Pipeline([...])
# final_logit_model.fit(...)

# classifier = ...

# for class_index, class_name in enumerate(...):
#     ...


## 13. Compare with a nonlinear model: Random Forest

The Logistic Regression model is linear in feature space.

A Random Forest can learn nonlinear thresholds and interactions. We evaluate it with the **same leave-one-subject-out rule** so the comparison is fair.

This cell may take a few seconds.


In [ ]:
# COMPLETE:
# Make a RandomForestClassifier:
#   n_estimators=50
#   random_state=42
#   class_weight="balanced"
#   n_jobs=-1
#
# Use cross_val_predict with LeaveOneGroupOut exactly as above.
#
# Calculate accuracy and macro F1.
#
# Make a two-row model_comparison DataFrame comparing
# Logistic Regression and Random Forest.

# rf_model = ...
# rf_pred_grouped = cross_val_predict(...)
# rf_accuracy = ...
# rf_macro_f1 = ...
# model_comparison = pd.DataFrame({...})
# display(model_comparison)


## 14. Reconnect this to the full raw-window sklearn Pipeline

For grouped cross-validation above, we calculated deterministic hand features once because it is much faster.

But we can still package the whole recipe into one sklearn-compatible object:

```text
raw 100×6 window
      ↓
IMUFeatureExtractor
      ↓
StandardScaler
      ↓
LogisticRegression
```

That is convenient when we eventually receive the hidden test windows.


In [ ]:
# Use your IMUFeatureExtractor class from Notebook 05.
# COMPLETE any missing pieces if needed.

class IMUFeatureExtractor(
    BaseEstimator,
    TransformerMixin,
):
    def __init__(self, channel_names=None, fs=50.0):
        self.channel_names = channel_names
        self.fs = fs

    def fit(self, X, y=None):
        # Store output feature names while fitting.
        self.feature_names_out_ = (
            extract_hand_features(
                X[:1],
                self.channel_names,
                self.fs,
            )
            .columns
            .to_numpy()
        )
        return self

    def transform(self, X):
        # COMPLETE:
        # return the hand-feature DataFrame converted to numpy
        pass

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_


# COMPLETE:
# Build raw_window_model:
# IMUFeatureExtractor -> StandardScaler -> LogisticRegression
#
# Fit it on X_windows, y_windows and predict the first five windows.


# Final checkpoint

You now have two dramatically different validation stories:

```text
Random windows
    → mixes the same people across train/test
    → optimistic

Leave one subject out
    → each validation subject is genuinely unseen
    → much closer to the final competition
```

Before you finish, answer:

1. Why is grouped validation harder than random-window validation?
2. Why is **macro F1** useful in addition to accuracy?
3. Which activity pair is confused most often?
4. Which anonymous subject was hardest to generalize to?
5. Did Random Forest improve on Logistic Regression?
6. Which hand-crafted features received the largest Logistic Regression coefficients?
7. Why is it okay to precompute our per-window hand features, but **not** okay to fit `StandardScaler` on everybody before cross-validation?
8. What information might a 1D CNN extract from the raw waveforms that our summary statistics discard?

The final hidden test subjects are still in the vault. 🔐
